# Wikidata Bootstrap: Simple Entity Linking

This notebook bootstraps a tiny local knowledge base from Wikidata and then uses exact string matching to annotate text with Wikidata URIs.

Default domain: living people currently holding a political office in Ireland.

## Important idea

The local KB is not hidden. It is just:

1. a SPARQL query,
2. the rows returned by Wikidata,
3. a little Python that groups those rows,
4. and a surface-form index built from the labels.

If you want a different KB, edit the `.sparql` query file or point `query_path` at another `.sparql` file, then rerun the notebook.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / 'src'))

from wikidata_lab.agentic_linking import AgenticLinkerTemplate
from wikidata_lab.candidate_retrieval import SimpleVectorStoreTemplate
from wikidata_lab.wikidata import (
    DEFAULT_BOOTSTRAP_QUERY_PATH,
    annotate_text,
    build_surface_form_index,
    fetch_knowledge_base_from_query,
    load_sparql_query,
)


In [ ]:
query_path = DEFAULT_BOOTSTRAP_QUERY_PATH
print(query_path)
query = load_sparql_query(query_path)
print(query)

## Change the KB here

Best practice: keep your SPARQL in a clearly named `.sparql` file.

For this notebook, edit the file printed above if you want to:

1. change country,
2. change occupation,
3. narrow to a different office,
4. or ask for a different domain entirely.

After editing the file, rerun the query-loading cell and then the next cell.

In [ ]:
raw_rows, kb = fetch_knowledge_base_from_query(query)
print(f'raw rows: {len(raw_rows)}')
print(f'entities in local KB: {len(kb)}')

In [ ]:
raw_rows[:5]

In [ ]:
kb[:5]

## Build a surface-form index

By default, each entity uses its Wikidata label as the surface form.

That keeps the baseline explicit and easy to understand.

If you want more surface forms, add aliases manually or extend the query.

In [ ]:
surface_form_index = build_surface_form_index(kb)
list(surface_form_index.items())[:5]

In [ ]:
sample_people = [entity['label'] for entity in kb[:3]]
sample_text = (
    f"{sample_people[0]} met {sample_people[1]} in Dublin while {sample_people[2]} "
    "spoke about the coalition."
)
annotations = annotate_text(sample_text, surface_form_index)
annotations

## Annotate the example file from the repository

This is the same baseline pipeline, but using a real example document from `data/` and passing its contents into `annotate_text(...)` as a function argument.

In [ ]:
example_path = REPO_ROOT / 'data' / 'example_current_irish_office_holders.txt'
example_text = example_path.read_text(encoding='utf-8')
print(example_text)

annotate_text(example_text, surface_form_index)

## Alias expansion exercise

Try adding extra surface forms to the local KB. This is one of the easiest first extensions.

Examples:

1. ASCII variants without accents
2. common shortened names
3. party-title variants
4. aliases fetched from Wikidata in a second query

In [ ]:
mary_lou_entity = next((entity for entity in kb if entity['label'] == 'Mary Lou McDonald'), None)
manual_aliases = {}
if mary_lou_entity is not None:
    manual_aliases[mary_lou_entity['uri']] = ['Mary Lou']

surface_form_index_with_aliases = build_surface_form_index(
    kb,
    extra_surface_forms_by_uri=manual_aliases,
)

annotate_text('Mary Lou addressed supporters.', surface_form_index_with_aliases)

## Vector retrieval template

The repository includes a dependency-free scaffold for vector retrieval.

To use it, plug in an embedding provider with `embed_texts(texts)`.

In [ ]:
class DemoEmbeddingProvider:
    def embed_texts(self, texts):
        vectors = []
        for text in texts:
            lowered = text.casefold()
            vectors.append([
                float('mary' in lowered),
                float('martin' in lowered),
                float(len(lowered)),
            ])
        return vectors

vector_store = SimpleVectorStoreTemplate(DemoEmbeddingProvider())
vector_store.index_entities(kb)
demo_mention = kb[0]['label']
vector_store.retrieve(demo_mention, top_k=5)

## Context-driven agentic linker template

This follows the same structure as the reference SNOMED project:

1. retrieve candidates,
2. extract local context,
3. build a prompt,
4. call an injected LLM backend.

In [ ]:
agent = AgenticLinkerTemplate()
candidate_list = vector_store.retrieve(demo_mention, top_k=3)
prompt = agent.build_prompt(
    full_text=f'{demo_mention} spoke in the Dail this afternoon.',
    mention_text=demo_mention,
    start=0,
    end=len(demo_mention),
    candidates=candidate_list,
)
print(prompt)

## Suggested student exercises

1. Change the SPARQL query to a new domain.
2. Add alias expansion.
3. Improve the spotter to handle punctuation and accents.
4. Add a better candidate retrieval stage.
5. Plug in an LLM backend and build a context-aware linker.
6. Create a tiny evaluation set and score your linker.